# Extration de Data avec Pandas




In [ ]:
#%pip install pandas

# import pandas as pd


# #que contiennent les en-têtes
# data = pd.read_csv("../data/en.openfoodfacts.org.products.csv")

# #import du fichier .csv

# data.head()

import pandas as pd

df = pd.read_csv("../data/en.openfoodfacts.org.products.csv", sep="\t", nrows=5)

colonnes = pd.DataFrame({
    "index": range(len(df.columns)),
    "nom_colonne": df.columns
})

display(colonnes)

df.to_csv("..\data\produits_col.csv", index=False)

<>:24: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:24: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Administrateur\AppData\Local\Temp\ipykernel_29532\755980363.py:24: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df.to_csv("..\data\produits_col.csv", index=False)


,index,nom_colonne
0,0,code
1,1,url
2,2,creator
3,3,created_t
4,4,created_datetime
...,...,...
206,206,sulphate_100g
207,207,nitrate_100g
208,208,acidity_100g
209,209,carbohydrates-total_100g


-> On recupere le nom des colonnes pour déterminer celle qui vont nous interresser pour le premier trie des données


In [21]:
#%pip install duckdb

import duckdb

result = duckdb.sql("""
SELECT code, product_name, countries, no_nutrition_data, brands, energy_100g, sugars_100g, salt_100g
FROM read_csv_auto('../data/en.openfoodfacts.org.products.csv', delim='\t')

""").df()

print(result)

# Code
# product_name
# countries
# no_nutrition_data
# brand
# energy_100g
# sugars_100g
# salt_100g

result.to_csv("..\data\produits_fr.csv", index=False)



<>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\Administrateur\AppData\Local\Temp\ipykernel_29532\744233579.py:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  result.to_csv("..\data\produits_fr.csv", index=False)


                                             code  \
0                                    000000000054   
1                                    000000000063   
2                                    000000000114   
3                                    000000000431   
4                                      0000000105   
...                                           ...   
4535548                             9999999999901   
4535549                             9999999999970   
4535550                             9999999999994   
4535551                             9999999999999   
4535552  9999999999999999999999999999999999996994   

                                 product_name          countries  \
0               Limonade artisanale a la rose              en:fr   
1                               M&amp;M white              en:fr   
2                                Chocolate n3             France   
3                              Pâte de fruits              en:fr   
4        Paleta gran re

In [36]:
import duckdb

duckdb.sql("""
COPY (
    SELECT DISTINCT countries_tags
    FROM read_csv_auto(
        '../data/en.openfoodfacts.org.products.csv',
        delim='\t'
    )
    WHERE countries IS NOT NULL
    ORDER BY countries
)
TO '../data/valeur_countries_tag.csv'
WITH (HEADER, DELIMITER ',');
""")

In [37]:
#pour chaque ligne de mon fichier valeur_countries.csv

#recupere la valeur ( un gros string)
#chaque valeur al interieur du string separé par ,
#eclate moi la chaine au , recupere la valeur met dans une liste tempo 
#ensuite pour cette meme liste verifie si les valeur sont deja dans la liste de valeur unique si pas ajoute sinon passe
#passe a la ligne suivante
#transforme ma liste de valeur unique en fichier csv

import pandas as pd

pays_uniques = set()

with open("../data/valeur_countries_tag.csv", "r", encoding="utf-8") as f:

    next(f)  # saute l'en-tête

    for ligne in f:

        ligne = ligne.strip()

        for pays in ligne.split(","):

            pays = pays.strip()

            if pays:
                pays_uniques.add(pays)

print(len(pays_uniques))




df_resultat = pd.DataFrame(
    sorted(pays_uniques),
    columns=["country"]
)

df_resultat.to_csv(
    "../data/liste_pays_uniques_tag.csv",
    index=False
)



# Lecture
df = pd.read_csv("../data/liste_pays_uniques_tag.csv")

# Supprime les guillemets
df["country"] = df["country"].astype(str).str.replace('"', '', regex=False)

# Supprime les espaces en début/fin
df["country"] = df["country"].str.strip()

# Supprime les doublons
df = df.drop_duplicates()

# Trie alphabétique
df = df.sort_values("country")

# Réinitialise l'index
df = df.reset_index(drop=True)

# Sauvegarde
df.to_csv("../data/liste_pays_uniques_nettoyee_tag.csv", index=False)

print(f"Nombre de valeurs uniques : {len(df)}")

818
Nombre de valeurs uniques : 402
